# Tri automatique des déchets — Entraînement du modèle CNN
**Sujet 11 — Mini-mémoire Deep Learning (M1 IA Data ISM)**

Ce notebook fait tout le pipeline :
1. Récupération et organisation du dataset TrashNet (6 classes)
2. Prétraitement et augmentation de données
3. Modèle par transfert d'apprentissage (MobileNetV2)
4. Entraînement en deux phases (tête seule, puis fine-tuning)
5. Évaluation (matrice de confusion, rapport par classe)
6. Sauvegarde du modèle pour l'outil de démo webcam

À exécuter sur **Google Colab** avec un runtime GPU (Exécution > Modifier le type d'exécution > GPU).


## 1. Installation et imports

In [ ]:
# Bibliothèques nécessaires (déjà présentes sur Colab pour la plupart)
!pip install -q split-folders

import os
import pathlib
import random
import shutil
import zipfile

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Reproductibilité : on fixe les graines aléatoires
seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

print("TensorFlow version :", tf.__version__)
print("GPU disponible :", tf.config.list_physical_devices('GPU'))


## 2. Récupération du dataset TrashNet

TrashNet contient 2527 images réparties en 6 classes : `cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`.
On le récupère depuis le dépôt GitHub officiel.

In [ ]:
# Téléchargement du dataset depuis le dépôt GitHub officiel de TrashNet
!git clone -q https://github.com/garythung/trashnet.git

# Les images compressées se trouvent dans trashnet/data/dataset-resized.zip
zip_path = "trashnet/data/dataset-resized.zip"
extract_path = "dataset_brut"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Le zip crée un sous-dossier dataset-resized/ avec les 6 classes
dataset_source = os.path.join(extract_path, "dataset-resized")
print("Classes trouvées :", sorted(os.listdir(dataset_source)))


## 3. Organisation en train / validation / test

On répartit les images : 70% entraînement, 15% validation, 15% test.
On utilise `splitfolders` pour garder une répartition stratifiée simple et reproductible.

In [ ]:
import splitfolders

dataset_split = "dataset_split"

splitfolders.ratio(
    dataset_source,
    output=dataset_split,
    seed=seed,
    ratio=(0.7, 0.15, 0.15)
)

class_names = sorted(os.listdir(os.path.join(dataset_split, "train")))
num_classes = len(class_names)
print("Classes :", class_names)
print("Nombre de classes :", num_classes)

# Comptage des images par classe et par split (utile pour le mémoire, section Données)
for split in ["train", "val", "test"]:
    print(f"\n--- {split} ---")
    for classe in class_names:
        chemin = os.path.join(dataset_split, split, classe)
        print(f"{classe:12s} : {len(os.listdir(chemin))} images")


## 4. Chargement des données et augmentation

On charge les images avec `image_dataset_from_directory`, puis on applique de l'augmentation
(rotation, zoom, flip, luminosité) uniquement sur le train pour améliorer la généralisation.
Important : la préparation des pixels utilise `preprocess_input` de MobileNetV2, pas une simple division par 255.

In [ ]:
img_size = (224, 224)
batch_size = 32

train_dir = os.path.join(dataset_split, "train")
val_dir = os.path.join(dataset_split, "val")
test_dir = os.path.join(dataset_split, "test")

train_data = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=img_size, batch_size=batch_size,
    label_mode="categorical", shuffle=True, seed=seed
)
val_data = tf.keras.utils.image_dataset_from_directory(
    val_dir, image_size=img_size, batch_size=batch_size,
    label_mode="categorical", shuffle=False
)
test_data = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=img_size, batch_size=batch_size,
    label_mode="categorical", shuffle=False
)

# On garde les vrais noms de classes dans l'ordre utilisé par Keras (important pour l'outil final)
class_names = train_data.class_names
print("Ordre des classes utilisé par le modèle :", class_names)

# Couche d'augmentation de données, appliquée seulement à l'entraînement
augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name="augmentation")

def preparer_train(image, label):
    image = augmentation(image)
    image = preprocess_input(image)
    return image, label

def preparer_eval(image, label):
    image = preprocess_input(image)
    return image, label

train_data = train_data.map(preparer_train).prefetch(tf.data.AUTOTUNE)
val_data = val_data.map(preparer_eval).prefetch(tf.data.AUTOTUNE)
test_data = test_data.map(preparer_eval).prefetch(tf.data.AUTOTUNE)


## 5. Gestion du déséquilibre entre classes

TrashNet n'est pas parfaitement équilibré (ex : peu d'images pour "trash"). On calcule des poids de classe
pour que le modèle ne néglige pas les classes minoritaires.

In [ ]:
# On récupère les labels du dossier train pour calculer les poids de classe
train_labels = []
for classe_index, classe in enumerate(class_names):
    chemin = os.path.join(train_dir, classe)
    train_labels += [classe_index] * len(os.listdir(chemin))

poids_classes = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=train_labels
)
class_weight_dict = dict(enumerate(poids_classes))
print("Poids par classe :", class_weight_dict)


## 6. Construction du modèle (transfert d'apprentissage — MobileNetV2)

On part d'un MobileNetV2 pré-entraîné sur ImageNet, dont on gèle les poids dans un premier temps.
On ajoute une petite "tête" de classification adaptée à nos 6 classes.

In [ ]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # on gèle le modèle de base pour la phase 1

entree = layers.Input(shape=(224, 224, 3))
x = base_model(entree, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
sortie = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(entree, sortie, name="tri_dechets_cnn")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 7. Phase 1 — Entraînement de la tête (base gelée)

In [ ]:
callbacks_phase1 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)
]

historique_phase1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase1
)


## 8. Phase 2 — Fine-tuning (dégel partiel du modèle de base)

On dégèle les dernières couches de MobileNetV2 et on continue l'entraînement avec un taux
d'apprentissage beaucoup plus faible, pour affiner sans détruire les poids pré-entraînés.

In [ ]:
base_model.trainable = True

# On ne dégèle que les 30 dernières couches, le reste du réseau reste gelé
for couche in base_model.layers[:-30]:
    couche.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("meilleur_modele.keras", monitor="val_loss", save_best_only=True)
]

historique_phase2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase2
)


## 9. Courbes d'apprentissage (à mettre dans le mémoire, section Résultats)

In [ ]:
def fusionner_historiques(h1, h2, cle):
    return h1.history[cle] + h2.history[cle]

accuracy = fusionner_historiques(historique_phase1, historique_phase2, "accuracy")
val_accuracy = fusionner_historiques(historique_phase1, historique_phase2, "val_accuracy")
loss = fusionner_historiques(historique_phase1, historique_phase2, "loss")
val_loss = fusionner_historiques(historique_phase1, historique_phase2, "val_loss")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(accuracy, label="Entraînement")
axes[0].plot(val_accuracy, label="Validation")
axes[0].axvline(x=len(historique_phase1.history["accuracy"]), color="gray", linestyle="--", label="Début fine-tuning")
axes[0].set_title("Exactitude (accuracy)")
axes[0].set_xlabel("Époque")
axes[0].legend()

axes[1].plot(loss, label="Entraînement")
axes[1].plot(val_loss, label="Validation")
axes[1].axvline(x=len(historique_phase1.history["loss"]), color="gray", linestyle="--", label="Début fine-tuning")
axes[1].set_title("Perte (loss)")
axes[1].set_xlabel("Époque")
axes[1].legend()

plt.tight_layout()
plt.savefig("courbes_apprentissage.png", dpi=150)
plt.show()


## 10. Évaluation finale sur le jeu de test

In [ ]:
perte_test, exactitude_test = model.evaluate(test_data)
print(f"Exactitude sur le test : {exactitude_test:.2%}")
print(f"Perte sur le test : {perte_test:.4f}")

# Prédictions sur tout le jeu de test pour le rapport détaillé
vraies_classes = []
predictions = []

for images, labels in test_data:
    preds = model.predict(images, verbose=0)
    predictions.extend(np.argmax(preds, axis=1))
    vraies_classes.extend(np.argmax(labels.numpy(), axis=1))

print("\nRapport de classification :\n")
print(classification_report(vraies_classes, predictions, target_names=class_names))


## 11. Matrice de confusion (visuel indispensable pour le mémoire)

In [ ]:
matrice = confusion_matrix(vraies_classes, predictions)

plt.figure(figsize=(7, 6))
sns.heatmap(matrice, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")
plt.title("Matrice de confusion — Tri des déchets")
plt.tight_layout()
plt.savefig("matrice_confusion.png", dpi=150)
plt.show()


## 12. Sauvegarde du modèle pour l'outil de démo

On sauvegarde le modèle final et la liste ordonnée des classes (essentielle : l'ordre doit être
identique à celui utilisé pendant l'entraînement pour que les prédictions restent correctes).

In [ ]:
model.save("modele_tri_dechets.keras")

with open("classes.txt", "w") as fichier:
    fichier.write("\n".join(class_names))

print("Modèle sauvegardé : modele_tri_dechets.keras")
print("Classes sauvegardées :", class_names)

# Pense à télécharger ces deux fichiers depuis Colab (icône dossier à gauche)
# et à les placer dans le même dossier que app_webcam.py
